In [0]:
%run /Workspace/Users/kaustuvdev8123@outlook.com/utilClass/Encryption


In [0]:
import uuid
import os
from delta.tables import DeltaTable
import time




base_path = "abfss://data@stdevwesteuropewrmp.dfs.core.windows.net"

read_path = base_path + "/bronze/hotel_weather_raw"

silver_write_path = base_path + "/silver/hotel_weather_processed"

schema_location_path = base_path + "/metadata/silver/schema"


encrypted_bronze = spark.readStream\
    .format("delta")\
    .option('inferSchema', True)\
    .load(read_path)

columns_to_be_decrypted = ("address", "city", "country", "name")

decrypted_bronze = Encrypter().decrypt_columns(
    encrypted_bronze, *columns_to_be_decrypted
)



('address', 'city', 'country', 'name')


In [0]:
import pyspark.sql.functions as F
# Trimming whitespace from string fields (e.g., using trim()).
columns_to_be_trimmed = ("name", "address", "city", "country", "wthr_date", "_rescued_data", "geoHash")

for column in columns_to_be_trimmed:
    decrypted_bronze = decrypted_bronze.withColumn(column, F.trim(F.col(column)))


In [0]:
# Replacing blanks with null values (e.g., using nullif()).
string_columns = ("name", "address", "city", "country", "wthr_date", "_rescued_data", "geoHash")
for columns in string_columns:
    decrypted_bronze = decrypted_bronze.withColumn(columns, F.nullif(F.col(columns), F.lit("")))

In [0]:
# Standardizing date formats (e.g., using to_date()).
decrypted_bronze = decrypted_bronze.withColumn("wthr_date", F.to_date(F.col("wthr_date"), "yyyy-MM-dd"))


In [0]:
# Casting data types where necessary (e.g., from string to int using cast()).
decrypted_bronze = decrypted_bronze.withColumn("id", F.col("id").cast("long"))


In [0]:
# Renaming columns if needed (e.g., using .withColumn("new_name", col("old_name")))
#Nothing need be changed

In [0]:
# Removing records with null values in critical fields (e.g., using filter() or dropna()).
critical_columns = ("name", "geohash", "latitude", "longitude", "wthr_date")
decrypted_bronze = decrypted_bronze.dropna(subset=critical_columns)

In [0]:
# Removing duplicate records using deduplication logic compatible with streaming (e.g., dropDuplicates(["id", "timestamp_column"])).
decrypted_bronze = decrypted_bronze.dropDuplicates(["id", "wthr_date"])

In [0]:
# Writing the table
columns_to_be_encrypted = ("address", "city", "country", "name")

encrypted_silver = Encrypter().encrypt_columns(
    decrypted_bronze, *columns_to_be_encrypted
)
encrypted_silver.writeStream\
    .format("delta")\
    .option("checkpointLocation", f"{base_path}/checkpoint/silver_checkpoint")\
    .outputMode("append")\
    .start(silver_write_path)